# 03: Time-Series Spending Forecasting
### Feature Engineering, Preventing Data Leakage, and Rolling-Window Benchmarking

Here we formulate next-month personal finance expenditure forecasting:
- **Feature Engineering**: spending aggregates, behavioral metrics, lag features, moving averages
- **Time-Series Split / Expanding Window**: Strict prevention of lookahead bias / data leakage
- **Model Comparison**:
  1. Naive Baseline (3-month moving average)
  2. Linear Regression (Ridge)
  3. Random Forest Regressor
  4. Gradient Boosting / XGBoost Regressor


In [ ]:
import os
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.forecasting import aggregate_monthly_features, evaluate_rolling_time_series, forecast_next_month
print("Loaded forecasting module.")


## 1. Feature Engineering: Monthly Aggregations & Lag Features


In [ ]:
feat_df = aggregate_monthly_features('../data/processed/cleaned_transactions.csv')
display(feat_df.tail(8))


## 2. Rolling Time-Series Cross Validation (No Data Leakage)
We train only on past months $t < T$ and test on month $T$ incrementally.


In [ ]:
bench_df, test_details = evaluate_rolling_time_series(feat_df)
display(bench_df)


## 3. Forecast Trajectory vs Actual Spending


In [ ]:
plt.figure(figsize=(12, 5))
months = test_details['months']
plt.plot(months, test_details['actuals'], 'o-', label='Actual Spending', color='black', linewidth=2.5)
plt.plot(months, test_details['rf'], 's--', label='Random Forest', color='#2563eb', alpha=0.85)
plt.plot(months, test_details['xgb'], '^--', label='XGBoost', color='#10b981', alpha=0.85)
plt.plot(months, test_details['naive'], 'x:', label='Naive Baseline (3m Avg)', color='#f59e0b', alpha=0.85)

plt.title("Rolling-Window Out-of-Sample Predictions vs Actual Spending", fontsize=13, fontweight='bold')
plt.xlabel("Month")
plt.ylabel("Spending (€)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 4. Next Month's Forecast with Confidence Interval


In [ ]:
fc = forecast_next_month()
print(f"Upcoming Month Point Forecast: €{fc['predicted_total']:,.2f}")
print(f"80% Prediction Interval:       €{fc['lower_bound']:,.2f} - {fc['upper_bound']:,.2f}")
print(f"Recent 3-Month Average:        €{fc['three_month_average']:,.2f}")
print(f"Expected Month-over-Month:     €{fc['mom_change_eur']:+,.2f} ({fc['mom_change_pct']}%)")
print("\nCategory Forecasts:")
for cat, val in fc['category_forecasts'].items():
    print(f"  {cat:30}: €{val:,.2f}")
